# Assignment Text summariser powered by LLM

In [2]:
#For colab
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
# Importing packages
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
from transformers import AdamW, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import evaluate

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Checking prerequisites

In [4]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU
PyTorch version: 2.2.2+cu121


In [5]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
import torch
print("Torch version:", torch.__version__)
print("Torch location:", torch.__file__)

import transformers
print("Transformers version:", transformers.__version__)

Torch version: 2.2.2+cu121
Torch location: c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\torch\__init__.py
Transformers version: 4.40.0


### Loading Dataset

In [7]:
dataset = load_from_disk("S:\Projects\Datasets\TextS\samsum_dataset")

#### Neural networks cannot process raw text directly. Text must first be converted into numerical representations. Modern LLMs achieve this using subword tokenization.

#### LLM's like gpt rely on tokenization methods like BPE, but for this dataset and T5 llm we will use SentencePiece

#### Lets take a look at token embedding and trannsformer, Text to text transfer transformer, Flan-T5-base (trained on 250M parameters), big enough to be called LLM

In [8]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Lets inspect the model

In [9]:
model.config

T5Config {
  "_name_or_path": "google/flan-t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "

In [10]:
model.shared

Embedding(32128, 768)

In [11]:
model.encoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseGatedActDense(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
            (wo): Linear(in_features=2048, out_features=768, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
      

In [12]:
model.decoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerCrossAttention(
          (EncDecAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=F

In [13]:
model.lm_head

Linear(in_features=768, out_features=32128, bias=False)

#### There is 1 difference in encoder and decoder as seen above, encoder dosent have cross attention layer. The encoder only reads the input sentence, and dosent need another sequence to attend to.

#### The decoder has already generated summary, but it also takes input from enncoder along with generated input which is cross verified while generating the optimal output. This process is also called self supervising learning.

### Also the tokenizer

In [14]:
tokenizer.vocab_size

32100

In [15]:
tokenizer.model_max_length

512

In [16]:
tokenizer.pad_token

'<pad>'

In [17]:
tokenizer.eos_token

'</s>'

In [18]:
sample = dataset["train"][0]["dialogue"]
print(sample)

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)


In [19]:
tokens = tokenizer.tokenize(sample)
print(tokens[:50])

['▁Amanda', ':', '▁I', '▁baked', '▁cookies', '.', '▁Do', '▁you', '▁want', '▁some', '?', '▁Jerry', ':', '▁Sure', '!', '▁Amanda', ':', '▁I', "'", 'll', '▁bring', '▁you', '▁tomorrow', '▁', ':', '-', ')']


In [20]:
ids = tokenizer.encode(sample)
print(ids[:50])

[21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1]


### model already has pretrained embedding lets extract one

In [21]:
embeddings = model.get_input_embeddings().weight
print(embeddings.shape)

torch.Size([32128, 768])


In [22]:
tokens = tokenizer.tokenize("hello")
print(tokens)

token = tokens[0]

token_id = tokenizer.convert_tokens_to_ids(token)

print(token)
print(token_id)

['▁hello']
▁hello
21820


#### The pretrainied embedding has 21820 as hello, i will make more sense when compared with surounding vectors and it will have closer semantic meaning

### T5 base dont need a positional embedding, its has Relative position, biased inside the attention mechanism

## Now lets take a look at attention block

In [23]:
print(model.encoder.block[0].layer[0])

T5LayerSelfAttention(
  (SelfAttention): T5Attention(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
    (relative_attention_bias): Embedding(32, 12)
  )
  (layer_norm): T5LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
)


In [24]:
attn = model.encoder.block[0].layer[0].SelfAttention
print(attn)

T5Attention(
  (q): Linear(in_features=768, out_features=768, bias=False)
  (k): Linear(in_features=768, out_features=768, bias=False)
  (v): Linear(in_features=768, out_features=768, bias=False)
  (o): Linear(in_features=768, out_features=768, bias=False)
  (relative_attention_bias): Embedding(32, 12)
)


In [25]:
print(attn.has_relative_attention_bias)

True


##### This shows that the model has relative attention type, rather than absolute that we see in classic gpt

In [26]:
print(attn.relative_attention_bias)

Embedding(32, 12)


#### Here 32 is the relative distance buckets with 12 attention heads, model learns distance rather than positions, which is a smarter approach

In [27]:
# Attention weights
attn.relative_attention_bias.weight

Parameter containing:
tensor([[ 3.3072e+00, -1.4124e+01,  2.2363e+00, -7.5515e+00,  8.4037e+00,
          5.4025e+00,  4.9113e-01,  2.5243e-01,  4.3401e+00,  6.6022e+00,
         -8.6801e+00, -2.5473e+01],
        [-2.5756e+01,  1.0481e+01,  8.4726e+00,  3.9471e+00,  9.8540e+00,
          1.7485e+00,  9.1644e+00,  6.1179e+00,  7.9472e+00, -4.2284e+00,
          2.8060e+00,  7.6758e+00],
        [-1.5956e+01,  8.7715e+00,  5.2965e+00,  4.5750e+00,  7.7746e+00,
          9.5001e-01,  8.6429e+00,  6.6384e+00,  7.5241e+00, -1.7510e+01,
          3.7001e+00,  8.0501e+00],
        [-1.5508e+01,  7.6623e+00,  4.6198e+00,  4.7793e+00,  6.7673e+00,
          1.9559e+00,  8.1014e+00,  6.7059e+00,  7.0760e+00, -1.9015e+01,
          3.9715e+00,  8.0325e+00],
        [-1.3945e+01,  7.0022e+00,  4.4271e+00,  4.7923e+00,  5.8716e+00,
          2.2086e+00,  7.6472e+00,  6.8344e+00,  6.7654e+00, -2.1642e+01,
          4.1278e+00,  7.9277e+00],
        [-1.5966e+01,  6.4181e+00,  4.3777e+00,  4.9517e+0

#### These are pretrained weights

# PreProcesssing

In [28]:
# Dataset in use
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [29]:
# Look at sample
dataset["train"][0]

{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

In [30]:
sample = dataset["train"][0]

prompt = f"""Summarize the following conversation.

Dialogue:
{sample['dialogue']}

Summary:"""

print(prompt)

Summarize the following conversation.

Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:


### Applying the above example to whole dataset, because for FLAN an instruction prompt will give out better results

In [31]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 64

In [32]:
def create_prompt(dialogue):
    return f"""Summarize the following conversation.

Dialogue:
{dialogue}

Summary:"""

In [33]:
def preprocess_function(sample):

    # Create instruction prompt
    prompt = create_prompt(sample["dialogue"])

    # Tokenize encoder input
    model_inputs = tokenizer(
        prompt,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    # Tokenize decoder target
    labels = tokenizer(
        sample["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    # Ignore PAD tokens during loss computation
    labels["input_ids"] = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [34]:
# apply to whole dataset
tokenized_dataset = dataset.map(
    preprocess_function,
    remove_columns=dataset["train"].column_names
)

In [35]:
# Check a sample
tokenized_dataset["train"][0]

{'input_ids': [12198,
  1635,
  1737,
  8,
  826,
  3634,
  5,
  5267,
  10384,
  10,
  21542,
  10,
  27,
  13635,
  5081,
  5,
  531,
  25,
  241,
  128,
  58,
  16637,
  10,
  10625,
  55,
  21542,
  10,
  27,
  31,
  195,
  830,
  25,
  5721,
  3,
  10,
  18,
  61,
  20698,
  10,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]}

## Fine tuning the data and model

In [36]:
# data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

In [37]:
# Get dataloader
train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=2,
    shuffle=True,
    collate_fn=data_collator
)

#### The collator creates the decoder_input_ids automatically,, this process is called "teacher forcing", at every step the decoder is shown the correct previous word, not its own prediction that can be wrong.

In [38]:
# Configuration
EPOCHS = 3
LEARNING_RATE = 5e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)


# Scheduler
num_training_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
# Run this in colab for faster training, (approx train time in collab: 1hr, approx train time on local GPU: 12hrs)
model.train()

for epoch in range(EPOCHS):

    total_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in progress_bar:

        # Move batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(**batch)

        # Compute loss
        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        # Track loss
        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Average Loss: {average_loss:.4f}")

In [ ]:
# For colab
#SAVE_PATH = "/content/drive/MyDrive/flan_t5_samsum"

#model.save_pretrained(SAVE_PATH)
#tokenizer.save_pretrained(SAVE_PATH)

#print("Model saved successfully!")

### Testing on unseen data

In [63]:
path = r"S:\Projects\Datasets\model_textS\flan_t5_samsumV1"

model = AutoModelForSeq2SeqLM.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained(path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [41]:
dialogue = """
Mira: Okay, we need to rethink the entire hypothesis. The data isn’t matching our model.
Leon: I’ve been saying that the entropy term is too weak. It collapses under noise.
Arjun: Or maybe the noise isn’t noise. Maybe it’s a pattern we haven’t recognised yet.
Mira: A hidden variable?
Leon: Possibly. But that means rewriting half the thesis.
Arjun: Better rewrite than defend something broken.
Mira: True. Let’s start with the core assumption: the system is stable.
Leon: Except it clearly isn’t.
Arjun: Unless stability is local, not global.
Mira: That… actually fits the anomalies.
"""

In [64]:
def create_prompt(dialogue):
    return f"""Summarize the following conversation.

Dialogue:
{dialogue}

Summary:"""

In [65]:
model.eval()

prompt = create_prompt(dialogue)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=384
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4,
        early_stopping=True
    )

summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(summary)

In [78]:
fresh_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

diff = (model.shared.weight - fresh_model.shared.weight).abs().mean()
print("Mean abs diff in embedding weights:", diff.item())

Mean abs diff in embedding weights: 8.617470741271973


In [79]:
for name, p in model.named_parameters():
    m = p.abs().max().item()
    if m > 50:
        print(name, m)

encoder.block.2.layer.1.DenseReluDense.wi_1.weight 130.97592163085938
encoder.block.8.layer.1.DenseReluDense.wi_1.weight 103.700927734375
decoder.block.11.layer.2.DenseReluDense.wi_1.weight 121.121826171875
